# GEOS 340 — Coastal Geography
## Weekly Python Companion Notebook

Each section below corresponds to one week of the course schedule and provides a short,
runnable Python demonstration of that week's core concept. All examples use **synthetic
or illustrative data** built with NumPy/Pandas/SciPy/Matplotlib so every cell runs
immediately with no downloads required.

**For actual lab assignments**, replace the synthetic data-generation lines in each
section with the real NOAA, USGS, or Sentinel/Landsat datasets provided on Canvas —
the analysis and plotting code is written to be adapted directly.

**Requirements:** `numpy`, `pandas`, `matplotlib`, `scipy` (all included with the
Anaconda distribution referenced in the syllabus).


### Setup
Run this cell first — it is used by every section below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, ndimage

plt.rcParams["figure.dpi"] = 100
np.set_printoptions(suppress=True)


---
## Week 1 (Aug 18–24) — Introduction to Coastal Geography
**Southern U.S. Coastal Focus:** Overview of the Gulf and Atlantic coasts of the southern United States; major coastal environments
**Lab/Activity:** Coastal map orientation; identify major U.S. Southern Coast regions

Plot reference coastal cities and classify them as Gulf Coast or Atlantic Coast by longitude.

In [ ]:
import matplotlib.pyplot as plt

regions = {
    "Galveston, TX": (-94.80, 29.30),
    "New Orleans, LA": (-90.07, 29.95),
    "Gulfport, MS": (-89.09, 30.37),
    "Mobile, AL": (-88.04, 30.69),
    "Tampa, FL": (-82.46, 27.95),
    "Miami, FL": (-80.19, 25.76),
    "Charleston, SC": (-79.93, 32.78),
    "Wilmington, NC": (-77.94, 34.23),
}

fig, ax = plt.subplots(figsize=(8, 6))
for name, (lon, lat) in regions.items():
    color = "tab:blue" if lon < -84 else "tab:orange"
    ax.scatter(lon, lat, c=color, s=80, edgecolor="k", zorder=3)
    ax.annotate(name, (lon, lat), textcoords="offset points", xytext=(6, 4), fontsize=9)

ax.set_title("Week 1 - Major U.S. Southern Coastal Regions (Gulf vs. Atlantic)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("week01_coastal_regions.png", dpi=100)

plt.show()

---
## Week 2 (Aug 25–31) — Coastal Systems: Processes, Scales & Energy
**Southern U.S. Coastal Focus:** Gulf of Mexico vs. Atlantic Ocean; regional differences in wave climate and coastline orientation
**Lab/Activity:** Coastal system conceptual model; Discussion 1 (R/Su)

Compare relative wave-energy environments using the deep-water wave energy density
equation, E = (1/8) ρ g H².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

coasts = ["Gulf of Mexico\n(low-energy)", "SE Atlantic\n(moderate-energy)"]
mean_wave_height_m = np.array([0.6, 1.2])
rho = 1025; g = 9.81
wave_energy_density = (1/8) * rho * g * mean_wave_height_m**2

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(coasts, wave_energy_density, color=["#1C7293", "#B85042"])
for i, v in enumerate(wave_energy_density):
    ax.text(i, v + 20, f"{v:,.0f} J/m2", ha="center")
ax.set_ylabel("Wave energy density (J/m2)")
ax.set_title("Week 2 - Relative Wave Energy: Gulf vs. Atlantic Coast")
plt.tight_layout()
plt.savefig("week02_wave_energy.png", dpi=100)

plt.show()

---
## Week 3 (Sep 1–7) — Waves, Wave Energy & Longshore Transport
**Southern U.S. Coastal Focus:** Texas and Florida beaches; wave-driven sediment transport
**Lab/Activity:** Calculate/interpret wave characteristics; shoreline sediment exercise

Solve the linear wave dispersion relation to shoal a swell from deep to shallow water,
then compute a simplified (CERC-style) relative longshore transport index as a function
of breaker angle.

In [ ]:
import numpy as np

def wave_length_deep(T):
    g = 9.81
    return g * T**2 / (2*np.pi)

def dispersion_solve(T, h, tol=1e-6, max_iter=100):
    L0 = wave_length_deep(T)
    L = L0
    for _ in range(max_iter):
        L_new = L0 * np.tanh(2*np.pi*h / L)
        if abs(L_new - L) < tol:
            break
        L = L_new
    return L

T = 8.0
for h in [20, 10, 5, 2]:
    L = dispersion_solve(T, h)
    C = L / T
    print(f"Depth {h:4.1f} m -> wavelength {L:5.2f} m, celerity {C:4.2f} m/s")

def longshore_transport_index(Hb, alpha_b_deg):
    alpha_b = np.radians(alpha_b_deg)
    return Hb**2.5 * np.sin(2*alpha_b)

for angle in [5, 15, 30, 45]:
    Q = longshore_transport_index(Hb=1.2, alpha_b_deg=angle)
    print(f"Breaker angle {angle:2d} deg -> relative transport index {Q:6.2f}")

---
## Week 4 (Sep 8–14) — Tides, Currents & Coastal Circulation
**Southern U.S. Coastal Focus:** Tidal regimes of Louisiana, Mississippi, Alabama, Georgia, and the Carolinas
**Lab/Activity:** Analyze tide-gauge data; tidal-range mapping

Build a synthetic tide curve from four harmonic constituents (M2, S2, K1, O1), compute
the form number, and classify the tidal regime (diurnal / semidiurnal / mixed).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

t = np.arange(0, 30*24, 0.5)
M2 = 0.55 * np.cos(2*np.pi*t/12.42)
S2 = 0.15 * np.cos(2*np.pi*t/12.00)
K1 = 0.20 * np.cos(2*np.pi*t/23.93)
O1 = 0.12 * np.cos(2*np.pi*t/25.82)
water_level = M2 + S2 + K1 + O1

df = pd.DataFrame({"hours": t, "water_level_m": water_level})

F = (0.20 + 0.12) / (0.55 + 0.15)
if F < 0.25:
    regime = "Semidiurnal"
elif F < 1.5:
    regime = "Mixed, mainly semidiurnal"
elif F < 3.0:
    regime = "Mixed, mainly diurnal"
else:
    regime = "Diurnal"
print(f"Form number F = {F:.2f} -> {regime}")

tidal_range = df["water_level_m"].max() - df["water_level_m"].min()
print(f"Modeled tidal range over 30 days: {tidal_range:.2f} m")

plt.figure(figsize=(9,4))
plt.plot(df["hours"][:240], df["water_level_m"][:240])
plt.title(f"Week 4 - Synthetic Tide Curve (5 days) - {regime}")
plt.xlabel("Hours"); plt.ylabel("Water level (m)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("week04_tide_curve.png", dpi=100)

plt.show()

---
## Week 5 (Sep 15–21) — Coastal Sediment & Shoreline Change
**Southern U.S. Coastal Focus:** Barrier-island sediment dynamics along the Gulf and Atlantic coasts
**Lab/Activity:** Historical shoreline-change analysis using aerial imagery; Discussion 2 (R/Su)

Compute shoreline change rates along a set of cross-shore transects using the **End
Point Rate (EPR)** method, the same logic used by USGS's DSAS tool.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n_transects = 20
baseline_position = np.zeros(n_transects)
years_between = 15
true_rate = np.linspace(-3.5, -0.5, n_transects)
shoreline_1 = baseline_position + np.random.normal(0, 1.5, n_transects)
shoreline_2 = shoreline_1 + true_rate*years_between + np.random.normal(0, 2, n_transects)

epr = (shoreline_2 - shoreline_1) / years_between

df = pd.DataFrame({
    "transect": np.arange(1, n_transects+1),
    "shoreline_date1_m": shoreline_1,
    "shoreline_date2_m": shoreline_2,
    "EPR_m_per_yr": epr
})
print(df.round(2).to_string(index=False))
print(f"\nMean shoreline change rate: {epr.mean():.2f} m/yr")
print(f"Transects classified as eroding (<0): {(epr<0).sum()} of {n_transects}")

plt.figure(figsize=(8,4))
colors = ["firebrick" if r < 0 else "seagreen" for r in epr]
plt.bar(df["transect"], df["EPR_m_per_yr"], color=colors)
plt.axhline(0, color="k", lw=0.8)
plt.xlabel("Transect ID"); plt.ylabel("Shoreline change rate (m/yr)")
plt.title("Week 5 - End Point Rate Shoreline Change Analysis")
plt.tight_layout()
plt.savefig("week05_shoreline_change.png", dpi=100)

plt.show()

---
## Week 6 (Sep 22–28) — Beaches, Dunes & Barrier Islands
**Southern U.S. Coastal Focus:** Padre Island, Dauphin Island, Mississippi barrier islands, Outer Banks
**Lab/Activity:** Barrier-island geomorphology mapping; **Exam 1 (Weeks 1–6)**

Analyze a synthetic cross-shore elevation profile to extract dune crest elevation,
dune height above Mean High Water, shoreline position, and foreshore slope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

distance = np.linspace(0, 150, 300)
elevation = (
    3.5 * np.exp(-((distance-20)**2)/40)
    - 0.03*distance
    + 1.0
)
mhw = 0.3

shoreline_idx = np.argmin(np.abs(elevation - mhw))
dune_crest_idx = np.argmax(elevation)

dune_height = elevation[dune_crest_idx] - mhw
foreshore_slope = np.polyfit(distance[shoreline_idx-10:shoreline_idx+10],
                             elevation[shoreline_idx-10:shoreline_idx+10], 1)[0]

print(f"Dune crest elevation: {elevation[dune_crest_idx]:.2f} m at {distance[dune_crest_idx]:.1f} m")
print(f"Dune height above MHW: {dune_height:.2f} m")
print(f"Shoreline position (MHW crossing): {distance[shoreline_idx]:.1f} m")
print(f"Approx. foreshore slope: {foreshore_slope:.3f} (rise/run)")

plt.figure(figsize=(8,4))
plt.plot(distance, elevation, color="saddlebrown")
plt.axhline(mhw, color="steelblue", ls="--", label="Mean High Water")
plt.fill_between(distance, elevation, mhw, where=(elevation>mhw), color="wheat", alpha=0.6)
plt.scatter(distance[dune_crest_idx], elevation[dune_crest_idx], color="darkgreen", zorder=5, label="Dune crest")
plt.xlabel("Cross-shore distance (m)"); plt.ylabel("Elevation (m)")
plt.title("Week 6 - Barrier-Island Beach Profile")
plt.legend()
plt.tight_layout()
plt.savefig("week06_beach_profile.png", dpi=100)

plt.show()

---
## Week 7 (Sep 29–Oct 5) — Estuaries & Coastal Lagoons
**Southern U.S. Coastal Focus:** Mississippi Sound, Mobile Bay, Galveston Bay, Tampa Bay, and Florida Bay
**Lab/Activity:** Estuary classification and salinity-gradient analysis

Model surface and bottom salinity along an estuary's along-axis distance, compute the
surface–bottom stratification, and classify the estuary as salt-wedge, partially mixed,
or well-mixed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

distance_km = np.linspace(0, 40, 41)
surface_salinity = 32 * (1 - np.exp(-distance_km/18))
bottom_salinity = 32 * (1 - np.exp(-distance_km/9))

stratification = bottom_salinity - surface_salinity
mean_strat = stratification[5:35].mean()

if mean_strat > 5:
    estuary_type = "Salt-wedge / highly stratified"
elif mean_strat > 1:
    estuary_type = "Partially mixed"
else:
    estuary_type = "Well-mixed"
print(f"Mean surface-bottom salinity difference: {mean_strat:.2f} PSU -> {estuary_type}")

plt.figure(figsize=(8,4))
plt.plot(distance_km, surface_salinity, label="Surface salinity")
plt.plot(distance_km, bottom_salinity, label="Bottom salinity")
plt.xlabel("Distance from river mouth (km)"); plt.ylabel("Salinity (PSU)")
plt.title(f"Week 7 - Estuarine Salinity Gradient ({estuary_type})")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("week07_estuary_salinity.png", dpi=100)

plt.show()

---
## Week 8 (Oct 6–12) — Coastal Wetlands, Marshes & Mangroves
**Southern U.S. Coastal Focus:** Louisiana wetlands, Mississippi Sound marshes, Georgia salt marshes, Florida mangroves
**Lab/Activity:** Wetland mapping using satellite imagery; Discussion 3 (R/Su)

Simulate Sentinel/Landsat-style NIR, Red, and Green reflectance bands, compute **NDVI**
and **NDWI**, and classify each pixel as upland, vegetated wetland, or open water.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)
h, w = 100, 100
nir = np.clip(np.random.normal(0.35, 0.1, (h, w)), 0, 1)
red = np.clip(np.random.normal(0.12, 0.05, (h, w)), 0, 1)
green = np.clip(np.random.normal(0.10, 0.05, (h, w)), 0, 1)

yy, xx = np.mgrid[0:h, 0:w]
creek = (np.abs(xx - (40 + 0.3*yy)) < 4)
nir[creek] = 0.05; red[creek] = 0.04; green[creek] = 0.08

ndvi = (nir - red) / (nir + red + 1e-6)
ndwi = (green - nir) / (green + nir + 1e-6)

classification = np.where(ndwi > 0.1, 2,
                  np.where(ndvi > 0.25, 1, 0))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
im0 = axes[0].imshow(ndvi, cmap="YlGn"); axes[0].set_title("NDVI"); plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(ndwi, cmap="Blues"); axes[1].set_title("NDWI"); plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(classification, cmap="viridis"); axes[2].set_title("Classification\n(0=upland,1=wetland,2=water)")
plt.suptitle("Week 8 - Wetland Mapping from Synthetic Multispectral Imagery")
plt.tight_layout()
plt.savefig("week08_wetland_ndvi_ndwi.png", dpi=100)

wetland_pct = 100*(classification==1).sum()/classification.size
water_pct = 100*(classification==2).sum()/classification.size
print(f"Wetland coverage: {wetland_pct:.1f}% | Open water: {water_pct:.1f}%")

plt.show()

---
## Week 9 (Oct 13–19) — Deltas & River–Coast Interactions
**Southern U.S. Coastal Focus:** Mississippi River Delta as a major case study; sediment supply, subsidence, land loss, and delta evolution
**Lab/Activity:** Mississippi Delta change analysis using DEMs and satellite imagery; **Exam 2 (Weeks 7–9)**

Compare two binary land/water masks representing a delta lobe at two time points, and
compute the net land-loss area and annual loss rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

np.random.seed(7)
size = 120
yy, xx = np.mgrid[0:size, 0:size]
land_t1 = ((xx-60)**2/45**2 + (yy-60)**2/35**2) < 1
land_t1 = land_t1.astype(float)

land_t2 = ndimage.binary_erosion(land_t1, iterations=3)
breach_mask = (np.random.rand(size, size) < 0.03) & land_t2
land_t2 = land_t2 & ~breach_mask
land_t2 = ndimage.binary_dilation(land_t2, iterations=1) & (land_t1.astype(bool) | land_t2)

pixel_area_km2 = 0.01
area_t1 = land_t1.sum() * pixel_area_km2
area_t2 = land_t2.sum() * pixel_area_km2
years = 20
loss_rate = (area_t1 - area_t2) / years

print(f"Delta land area (Time 1): {area_t1:.2f} km^2")
print(f"Delta land area (Time 2): {area_t2:.2f} km^2")
print(f"Net land loss over {years} yrs: {area_t1-area_t2:.2f} km^2 ({loss_rate:.3f} km^2/yr)")

change_map = np.zeros((size, size))
change_map[land_t1.astype(bool) & ~land_t2] = -1
change_map[~land_t1.astype(bool) & land_t2] = 1

plt.figure(figsize=(6,6))
plt.imshow(change_map, cmap="RdYlGn", vmin=-1, vmax=1)
plt.title("Week 9 - Delta Land Change (red = loss, green = gain)")
plt.axis("off")
plt.tight_layout()
plt.savefig("week09_delta_land_change.png", dpi=100)

plt.show()

---
## Week 10 (Oct 20–26) — Hurricanes, Storm Surge & Coastal Hazards
**Southern U.S. Coastal Focus:** Katrina, Rita, Ike, Harvey, Michael, Ian, and other Southern Coast hurricanes
**Lab/Activity:** Storm-track and storm-surge mapping; **Project Topics Due**

A simplified teaching approximation of storm surge height as a function of central
pressure, shelf width, and forward speed — illustrating why a slower storm over a wide
shelf can outsurge a faster, more intense storm over a narrow shelf.

*Note: this is a simplified teaching relationship, not an operational surge model
(e.g., SLOSH/ADCIRC) — useful for building intuition about the controlling factors.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def estimate_surge_m(central_pressure_hpa, shelf_width_km, forward_speed_kt):
    pressure_deficit = 1013 - central_pressure_hpa
    base_surge = 0.01 * pressure_deficit
    shelf_factor = 1 + shelf_width_km / 100.0
    speed_factor = 1 + max(0, (15 - forward_speed_kt))/30
    return base_surge * shelf_factor * speed_factor * 3

storms = pd.DataFrame({
    "storm": ["Fast/Cat3, narrow shelf", "Slow/Cat2, wide shelf", "Historical Ike-like"],
    "central_pressure_hpa": [946, 972, 950],
    "shelf_width_km": [15, 120, 100],
    "forward_speed_kt": [18, 6, 10],
})
storms["est_surge_m"] = storms.apply(
    lambda r: estimate_surge_m(r.central_pressure_hpa, r.shelf_width_km, r.forward_speed_kt), axis=1)
print(storms.round(2).to_string(index=False))

plt.figure(figsize=(6,4))
plt.bar(storms["storm"], storms["est_surge_m"], color="#B85042")
plt.ylabel("Estimated surge (m)")
plt.title("Week 10 - Simplified Storm Surge Comparison")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("week10_storm_surge.png", dpi=100)

plt.show()

---
## Week 11 (Oct 27–Nov 2) — Sea-Level Rise, Subsidence & Coastal Flooding
**Southern U.S. Coastal Focus:** Louisiana subsidence; Florida and Charleston vulnerability; recurrent tidal flooding
**Lab/Activity:** Analyze sea-level and elevation datasets; Discussion 4 (R/Su)

Fit a linear trend to a synthetic annual mean sea-level record, project it forward to
2070, and apply a simple "bathtub" flood model to a synthetic coastal DEM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

years = np.arange(1990, 2025)
np.random.seed(3)
sea_level_mm = (years-1990)*4.2 + np.random.normal(0, 8, len(years))

slope, intercept, r, p, se = stats.linregress(years, sea_level_mm)
print(f"Observed trend: {slope:.2f} mm/yr (R^2={r**2:.2f})")

future_years = np.arange(2025, 2071)
projected_mm = intercept + slope*future_years
projected_m_2070 = (projected_mm[-1] - sea_level_mm[0]) / 1000
print(f"Projected relative sea-level rise by 2070: {projected_m_2070:.2f} m")

size = 100
yy, xx = np.mgrid[0:size, 0:size]
dem = 0.02*xx + 0.3*np.sin(yy/12)
flood_extent = dem < projected_m_2070

fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot(years, sea_level_mm, "o", ms=3, label="Observed (synthetic)")
axes[0].plot(np.concatenate([years, future_years]),
             intercept + slope*np.concatenate([years, future_years]), "r-", label="Trend")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Relative sea level (mm)")
axes[0].legend(); axes[0].set_title("Week 11a - Sea-Level Trend & Projection")

axes[1].imshow(dem, cmap="terrain")
axes[1].contourf(flood_extent, levels=[0.5,1], colors=["deepskyblue"], alpha=0.6)
axes[1].set_title(f"Week 11b - Bathtub Flood Model\n(+{projected_m_2070:.2f} m by 2070)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("week11_slr_flooding.png", dpi=100)

plt.show()

---
## Week 12 (Nov 3–9) — Human Modification of Coastal Environments
**Southern U.S. Coastal Focus:** Ports, seawalls, levees, navigation channels, beach nourishment, and coastal development
**Lab/Activity:** Compare engineered vs. natural shorelines

Statistically compare shoreline change rates for engineered vs. natural shoreline
segments using an independent-samples t-test and a box plot.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(11)
n = 15
natural_rate = np.random.normal(-1.8, 0.8, n)
engineered_rate = np.random.normal(-0.3, 0.5, n)

df = pd.DataFrame({
    "segment": np.arange(1, n+1),
    "natural_m_per_yr": natural_rate,
    "engineered_m_per_yr": engineered_rate
})
print(df.describe()[["natural_m_per_yr", "engineered_m_per_yr"]].round(2))

t, p = stats.ttest_ind(natural_rate, engineered_rate)
print(f"\nt-test natural vs engineered shoreline change: t={t:.2f}, p={p:.4f}")

plt.figure(figsize=(6,4))
plt.boxplot([natural_rate, engineered_rate], tick_labels=["Natural shoreline", "Engineered shoreline"])
plt.axhline(0, color="k", lw=0.8)
plt.ylabel("Shoreline change rate (m/yr)")
plt.title("Week 12 - Engineered vs. Natural Shoreline Change")
plt.tight_layout()
plt.savefig("week12_engineered_vs_natural.png", dpi=100)

plt.show()

---
## Week 13 (Nov 10–16) — Coastal Ecosystems & Environmental Change
**Southern U.S. Coastal Focus:** Gulf Coast fisheries, oyster reefs, seagrass, coral reefs, mangroves, and salt marshes
**Lab/Activity:** Coastal habitat classification and change detection; **Exam 3 (Weeks 10–13)**

Build a land-cover **transition matrix** between two classified habitat maps to
quantify how much seagrass and oyster reef area converted to open water.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(5)
classes = ["Open Water", "Seagrass", "Oyster Reef", "Salt Marsh", "Upland"]
size = 80
t1 = np.random.choice(len(classes), size=(size,size), p=[0.35,0.2,0.1,0.25,0.1])
t2 = t1.copy()
decline_mask = (np.isin(t1, [1,2])) & (np.random.rand(size,size) < 0.25)
t2[decline_mask] = 0

transition = pd.crosstab(
    pd.Series(t1.ravel(), name="Time 1").map(dict(enumerate(classes))),
    pd.Series(t2.ravel(), name="Time 2").map(dict(enumerate(classes)))
)
print("Transition matrix (pixel counts):\n")
print(transition)

seagrass_loss_pct = 100 * decline_mask[t1==1].sum() / (t1==1).sum()
reef_loss_pct = 100 * decline_mask[t1==2].sum() / (t1==2).sum()
print(f"\nSeagrass area lost to open water: {seagrass_loss_pct:.1f}%")
print(f"Oyster reef area lost to open water: {reef_loss_pct:.1f}%")

---
## Week 14 (Nov 17–23) — Coastal Urbanization, Tourism & Socioeconomic Geography
**Southern U.S. Coastal Focus:** Miami, Tampa, New Orleans, Houston/Galveston, Charleston, and other rapidly developing coastal cities
**Lab/Activity:** Coastal population and development mapping

Compare metro-area population growth (1990–2020) across five Southern U.S. coastal
cities as a proxy for coastal development pressure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(21)
cities = ["Miami", "Tampa", "New Orleans", "Houston/Galveston", "Charleston"]
pop_1990 = np.array([1937, 2068, 1239, 3711, 507])
pop_2020 = np.array([2716, 3175, 1271, 7066, 799])

growth_pct = 100*(pop_2020 - pop_1990)/pop_1990
df = pd.DataFrame({"city": cities, "pop_1990_k": pop_1990, "pop_2020_k": pop_2020, "growth_pct": growth_pct})
print(df.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(cities))
ax.bar(x-0.2, pop_1990, width=0.4, label="1990")
ax.bar(x+0.2, pop_2020, width=0.4, label="2020")
ax.set_xticks(x); ax.set_xticklabels(cities, rotation=20, ha="right")
ax.set_ylabel("Metro population (thousands)")
ax.set_title("Week 14 - Coastal Metro Population Growth, 1990-2020")
ax.legend()
plt.tight_layout()
plt.savefig("week14_population_growth.png", dpi=100)

plt.show()

---
## Week 15 (Nov 24–30) — Coastal Management, Restoration & Adaptation
**Southern U.S. Coastal Focus:** Louisiana Coastal Master Plan, Mississippi/Alabama restoration, Florida resilience planning, and Atlantic Coast management
**Lab/Activity:** Evaluate competing coastal-management strategies

A simple **weighted multi-criteria decision analysis (MCDA)** comparing seawalls,
wetland restoration, managed retreat, and beach nourishment across cost, protection,
ecological benefit, and adaptability to sea-level rise.

In [ ]:
import pandas as pd

strategies = ["Seawall", "Wetland Restoration", "Managed Retreat", "Beach Nourishment"]

scores = pd.DataFrame({
    "Cost (lower better)":     [2, 4, 5, 2],
    "Protection Level":        [5, 3, 2, 3],
    "Ecological Benefit":      [1, 5, 4, 2],
    "Adaptability to SLR":     [2, 4, 5, 2],
}, index=strategies)

weights = pd.Series({
    "Cost (lower better)": 0.25,
    "Protection Level": 0.30,
    "Ecological Benefit": 0.20,
    "Adaptability to SLR": 0.25,
})

weighted_score = (scores * weights).sum(axis=1)
result = pd.DataFrame({"Weighted Score": weighted_score}).sort_values("Weighted Score", ascending=False)
print(scores)
print("\nWeights:\n", weights)
print("\nRanked strategies:\n", result.round(2))

---
## Week 16 (Dec 1–7) — Future Coasts & Student Presentations
**Southern U.S. Coastal Focus:** Climate change, sea-level rise, adaptation, resilience, and the future of the Southern U.S. Coast
**Lab/Activity:** Final project presentations; **Final Project Due**

A reusable **starter template** for the final GIS research project: swap in your own
CSV/NOAA/USGS dataset, adapt the analysis function, and generate your summary figure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_project_data(csv_path=None):
    if csv_path:
        return pd.read_csv(csv_path)
    years = np.arange(2000, 2024)
    demo_variable = np.cumsum(np.random.normal(0.5, 1.0, len(years)))
    return pd.DataFrame({"year": years, "demo_variable": demo_variable})

def analyze(df):
    trend = df["demo_variable"].diff().mean()
    print(f"Average annual change: {trend:.2f} units/yr")
    return trend

def plot_results(df, trend):
    plt.figure(figsize=(7,4))
    plt.plot(df["year"], df["demo_variable"], marker="o")
    plt.title(f"Week 16 - Final Project Template (avg change: {trend:.2f}/yr)")
    plt.xlabel("Year"); plt.ylabel("Variable of interest")
    plt.tight_layout()
    plt.savefig("week16_final_project_template.png", dpi=100)

np.random.seed(0)
data = load_project_data()
trend = analyze(data)
plot_results(data, trend)

plt.show()